<h1 style="color:#4A0072; border-bottom:3px solid #6A1B9A; padding-bottom:8px;"> Reconhecimento de Objetos em Tempo Real </h1>

<h2 style="color:#4A0072;">📖 Como Funciona a Detecção de Objetos em Tempo Real?</h2>

A **detecção de objetos** é uma das tarefas fundamentais da Visão Computacional. Diferente da *classificação*, que apenas responde "o que é", a detecção responde **"o que é"** e **"onde está"**, desenhando caixas delimitadoras (*bounding boxes*) ao redor dos objetos encontrados.

<h3 style="color:#6A1B9A;">🔧 As Ferramentas que Vamos Usar</h3>
<ul>
<li><b>OpenCV:</b> Biblioteca de Visão Computacional que nos permite <i>capturar vídeo da webcam</i>, manipular frames e exibir resultados em tempo real.</li>
<li><b>MediaPipe (Google):</b> Framework de IA do Google com modelos pré-treinados otimizados para tarefas de visão, como detecção de objetos, poses, mãos, rostos, etc.</li>
</ul>

<h3 style="color:#6A1B9A;">🧠 O Modelo: EfficientDet Lite0 (float32)</h3>
<ul>
<li><b>Arquitetura:</b> EfficientDet é uma família de detectores de objetos eficientes desenvolvida pelo Google. A versão <i>Lite0</i> é otimizada para dispositivos com recursos limitados.</li>
<li><b>Precisão float32:</b> Utilizamos a versão com pesos em <b>ponto flutuante de 32 bits</b>, que oferece maior precisão nas detecções comparada às versões quantizadas (int8/float16).</li>
<li><b>Treinamento:</b> O modelo foi pré-treinado no dataset <b>COCO</b>, capaz de reconhecer <b>80 categorias</b> de objetos do cotidiano (pessoas, carros, animais, utensílios, etc).</li>
</ul>

<h3 style="color:#6A1B9A;">⚙️ O Pipeline</h3>
<ol>
<li>A <b>webcam</b> captura frames continuamente (via OpenCV)</li>
<li>Cada frame é convertido para o formato <b>MediaPipe Image</b></li>
<li>O <b>timestamp</b> do frame é calculado via <code>cv2.getTickCount()</code></li>
<li>O <b>Object Detector</b> do MediaPipe analisa o frame com contexto temporal e retorna as detecções</li>
<li>Desenhamos as <b>bounding boxes</b> e os <b>rótulos</b> sobre o frame</li>
<li>O resultado é exibido em uma <b>janela em tempo real</b></li>
</ol>

<div style="padding:14px 18px; border-radius:8px; border-left:4px solid #6A1B9A; margin:10px 0;">
💡 <b>Dica:</b> A grande vantagem do MediaPipe é a <i>simplicidade</i>: com poucas linhas de código, conseguimos um sistema de detecção de objetos funcional e em tempo real!
</div>

<br>
<h2 style="color:#4A0072;">📥 Download do Modelo Pré-Treinado</h2>

Vamos baixar o modelo **EfficientDet Lite0** na versão **float32** diretamente dos servidores do Google.

O arquivo `.tflite` será salvo na mesma pasta deste notebook:

In [3]:
import urllib.request
import os

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float32/1/efficientdet_lite0.tflite"
MODEL_PATH = "efficientdet_lite0.tflite"

if not os.path.exists(MODEL_PATH):
    print("Baixando o modelo EfficientDet Lite0 (float32)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"Modelo salvo em: {MODEL_PATH}")
else:
    print(f"Modelo já existe: {MODEL_PATH}")

print(f"Tamanho do arquivo: {os.path.getsize(MODEL_PATH) / (1024*1024):.1f} MB")

Modelo já existe: efficientdet_lite0.tflite
Tamanho do arquivo: 13.2 MB


<br>
<h2 style="color:#4A0072;">🔧 Imports e Configuração</h2>

Importamos as bibliotecas e definimos os aliases do MediaPipe Tasks, seguindo o mesmo padrão do script de reconhecimento:

In [4]:
import cv2
import mediapipe as mp
import numpy as np

# Aliases do MediaPipe Tasks (mesmo padrão do script)
BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

print("Imports carregados com sucesso!")

Imports carregados com sucesso!


<br>
<h2 style="color:#4A0072;">🎥 Detecção em Tempo Real com a Webcam</h2>

Agora vem a parte principal! O código abaixo segue o mesmo padrão do script `webcam_recog.py`:

<h3 style="color:#6A1B9A;">Passo a passo do loop:</h3>
<ol>
<li><b>Configura</b> o detector e abre a webcam com resolução 640×480</li>
<li><b>Captura</b> e <b>espelha</b> cada frame (<code>cv2.flip</code>)</li>
<li><b>Converte</b> BGR → RGB e cria o <code>mp.Image</code></li>
<li><b>Calcula o timestamp</b> com <code>cv2.getTickCount()</code></li>
<li><b>Detecta</b> objetos com <code>detect_for_video()</code></li>
<li><b>Desenha</b> bounding boxes e rótulos no frame</li>
<li><b>Exibe</b> o resultado — pressione <b>Q</b> para sair</li>
</ol>

<div style="padding:14px 18px; border-radius:8px; border-left:4px solid #6A1B9A; margin:10px 0;">
⚠️ <b>Importante:</b> Pressione a tecla <b>Q</b> para encerrar a janela de vídeo. Se a janela travar, reinicie o kernel do notebook.
</div>

In [7]:
# --- Cores para desenhar (em BGR) ---
COR_CAIXA = (154, 27, 106)     # roxo (#6A1B9A em BGR)
COR_TEXTO_FUNDO = (72, 0, 74)  # roxo escuro (#4A0072 em BGR)
COR_TEXTO = (255, 255, 255)    # branco

def desenhar_deteccoes(frame, resultado):
    """Desenha as bounding boxes e rótulos no frame."""
    for deteccao in resultado.detections:
        bbox = deteccao.bounding_box
        x, y, w, h = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height

        categoria = deteccao.categories[0]
        nome = categoria.category_name
        confianca = categoria.score
        texto = f"{nome} ({confianca:.0%})"

        # Retângulo da bounding box
        cv2.rectangle(frame, (x, y), (x + w, y + h), COR_CAIXA, 3)

        # Fundo do rótulo
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(frame, (x, y - th - 10), (x + tw + 8, y), COR_TEXTO_FUNDO, -1)

        # Texto do rótulo
        cv2.putText(frame, texto, (x + 4, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, COR_TEXTO, 2)

    return frame


# --- Configuração do detector ---
options = ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    max_results=5,
    score_threshold=0.5,
)

# --- Webcam ---
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Iniciando detecção de objetos... Pressione 'q' para sair.")

with ObjectDetector.create_from_options(options) as detector:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        # O MediaPipe requer o formato RGB
        # Obs: OpenCV usa BGR por padrão
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Timestamp em milissegundos (necessário para o modo VIDEO)
        timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)

        # Realiza a detecção
        resultado = detector.detect_for_video(mp_image, timestamp_ms)

        # Desenha os resultados no frame original
        frame_anotado = desenhar_deteccoes(frame, resultado)

        cv2.imshow('Deteccao de Objetos - MediaPipe', frame_anotado)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("Webcam encerrada.")

Iniciando detecção de objetos... Pressione 'q' para sair.
Webcam encerrada.


<br>
<h2 style="color:#4A0072;">📝 Resumo</h2>

<div style="padding:14px 18px; border-radius:8px; border-left:4px solid #6A1B9A; margin:10px 0;">
💡 <b>O que aprendemos:</b><br><br>
✅ O <b>OpenCV</b> é responsável por <i>capturar</i> os frames da webcam e <i>exibir</i> o resultado visual.<br>
✅ O <b>MediaPipe</b> fornece o modelo de IA (<i>EfficientDet Lite0 float32</i>) que realiza a detecção propriamente dita.<br>
✅ No modo <b>VIDEO</b>, o detector usa <i>contexto temporal</i> entre frames, resultando em bounding boxes mais estáveis.<br>
✅ O <b>timestamp</b> é calculado com <code>cv2.getTickCount() / cv2.getTickFrequency() * 1000</code> e passado ao <code>detect_for_video()</code>.<br>
✅ Combinando as duas ferramentas, criamos um sistema de <b>detecção de objetos em tempo real</b> com poucas linhas de código!
</div>